# Galvo Distortion Calibration

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.

The galvo mirrors steer the laser, and the beam lands slightly off from where it was told to go, more so towards the edges of the field. The calibration burns a grid of dots at known commanded positions, measures where each dot actually appeared, and saves the correction map.

A camera picture is taken before and after the calibration so the burned grid can be compared side by side.

> **Warning:** the calibration **fires the laser** and permanently marks whatever is under it. Put a sacrificial piece in place first. It takes minutes, and disconnecting cancels it.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.instruments.health_pb2 import HealthRequest
from femtum_sdk.api import InstrumentApi

api_host = "localhost:5210"

with InstrumentApi(host=api_host) as api:
    health_response = api.health.GetHealth(HealthRequest())
    print(health_response)


## Camera Helpers

`capture_picture` grabs a PNG from the laser head camera at the full sensor resolution reported by the camera. `show_before_after` renders both pictures one after the other, exactly as the camera sent them (no resizing or re-encoding), so each one can be downloaded at full resolution from the cell output (right-click → *Save Image As…*).

In [ ]:
from IPython.display import Image, Markdown, display

from femtum_sdk.api.contracts.v1.dto.camera_pb2 import CameraInfo
from femtum_sdk.api.contracts.v1.dto.image_pb2 import Image as ImageDto, ImageFormat, ImageSize
from femtum_sdk.api.contracts.v1.instruments.camera_pb2 import CaptureImageRequest, ReadCameraInfoRequest


def capture_picture(api: InstrumentApi) -> bytes:
    camera_info: CameraInfo = api.laser_head_camera.ReadCameraInfo(ReadCameraInfoRequest())
    image: ImageDto = api.laser_head_camera.CaptureImage(CaptureImageRequest(
        format=ImageFormat.IMAGE_FORMAT_PNG,
        size=ImageSize(width=camera_info.max_frame_size.width, height=camera_info.max_frame_size.height),
    ))
    return image.data


def show_before_after(before_png: bytes, after_png: bytes) -> None:
    for title, picture_png in (("Before", before_png), ("After", after_png)):
        display(Markdown(f"**{title}**"))
        display(Image(data=picture_png, format="png"))


## Take the "Before" Picture

Make sure the sacrificial piece is in focus (see `laser_head_camera_autofocus.ipynb`) so the burned dots are visible on the "after" picture.

In [ ]:
with InstrumentApi(host=api_host) as api:
    before_full_calibration = capture_picture(api)

display(Image(data=before_full_calibration, format="png"))


## Perform a Full Calibration

> **Warning:** this cell **fires the laser**.

The full calibration builds the correction map from scratch. Run it after the optics are changed or serviced, or when no calibration exists yet.

- `grid_size` is the number of dots per side (11 → 11 × 11 dots).
- `grid_spacing` is optional; the daemon picks a spacing covering the field when it is omitted.
- `energy_per_pulse` must fall inside the range of the laser energy calibration.

The laser is a safety switch that no other operation flips for you, so the cell enables it first.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import EnergyMeasure, Nanojoules
from femtum_sdk.api.contracts.v1.instruments.femtosecond_laser_pb2 import EnableLaserRequest, IsEnabledRequest
from femtum_sdk.api.contracts.v1.instruments.galvo_distortion_calibration_pb2 import (
    PerformFullCalibrationRequest,
    PerformFullCalibrationResponse,
)
from femtum_sdk.api.contracts.v1.instruments.laser_execution_pb2 import LaserExecutionOptions

with InstrumentApi(host=api_host) as api:
    api.femtosecond_laser.Enable(EnableLaserRequest(enable=True))
    assert api.femtosecond_laser.IsEnabled(IsEnabledRequest()).enabled

    full_calibration: PerformFullCalibrationResponse = api.galvo_distortion_calibration.PerformFullCalibration(
        PerformFullCalibrationRequest(
            options=LaserExecutionOptions(energy_per_pulse=EnergyMeasure(value=100, unit=Nanojoules)),
            grid_size=11,
        )
    )

    after_full_calibration = capture_picture(api)

show_before_after(before_full_calibration, after_full_calibration)


## Inspect the Measured Distortion

Each grid point pairs the commanded (`nominal`) position with the `measured` one. `radial_error` is the distance between them, in `points_unit`.

In [ ]:
from femtum_sdk.api.contracts.v1.dto.galvo_distortion_calibration_pb2 import GalvoDistortionCalibrationAnalysis
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import AngleMeasureUnit, DistanceMeasureUnit


def print_analysis(analysis: GalvoDistortionCalibrationAnalysis) -> None:
    distortion = analysis.distortion
    unit = DistanceMeasureUnit.Name(distortion.points_unit)
    radial_errors = [point.radial_error for point in distortion.points]
    offset = distortion.offset

    print(f"Calibration ID: {analysis.id}")
    print(f"Parent full calibration ID: {analysis.parent_full_calibration_id or '-'}")
    print(f"Grid points: {len(radial_errors)}")
    print(f"Mean radial error: {sum(radial_errors) / len(radial_errors):.3f} {unit}")
    print(f"Max radial error: {max(radial_errors):.3f} {unit}")
    print(
        f"Offset: x={offset.origin_x.value:.3f} {DistanceMeasureUnit.Name(offset.origin_x.unit)}, "
        f"y={offset.origin_y.value:.3f} {DistanceMeasureUnit.Name(offset.origin_y.unit)}, "
        f"orientation={offset.orientation.value:.4f} {AngleMeasureUnit.Name(offset.orientation.unit)}"
    )


print_analysis(full_calibration.analysis)


## Perform an Offset Calibration (Optional)

> **Warning:** this cell **fires the laser**. Move the sacrificial piece to a fresh area first, otherwise the new dots land on top of the previous grid.

The offset calibration is the quick version: it burns a small grid (3 × 3 dots) and only corrects where the field sits and how it is rotated, keeping the correction map of a full calibration. Use it when the work has shifted but the optics have not changed. Without `parent_full_calibration_id` it builds on the most recent full calibration.

In [ ]:
from femtum_sdk.api.contracts.v1.instruments.galvo_distortion_calibration_pb2 import (
    PerformOffsetCalibrationRequest,
    PerformOffsetCalibrationResponse,
)

with InstrumentApi(host=api_host) as api:
    before_offset_calibration = capture_picture(api)

    api.femtosecond_laser.Enable(EnableLaserRequest(enable=True))
    assert api.femtosecond_laser.IsEnabled(IsEnabledRequest()).enabled

    offset_calibration: PerformOffsetCalibrationResponse = api.galvo_distortion_calibration.PerformOffsetCalibration(
        PerformOffsetCalibrationRequest(
            options=LaserExecutionOptions(energy_per_pulse=EnergyMeasure(value=100, unit=Nanojoules)),
            parent_full_calibration_id=full_calibration.analysis.id,
            grid_size=3,
        )
    )

    after_offset_calibration = capture_picture(api)

show_before_after(before_offset_calibration, after_offset_calibration)
print_analysis(offset_calibration.analysis)
